# 심장 마비 발병 분류(1)
교과서 **222쪽** · 부록 활동 (힌트 버전)

환자의 건강 정보로 **심장 마비 발병 여부(`output`)**를 맞히는 **분류** 모델을 만듭니다.

| 오늘 할 일 | 힌트 키워드 |
|---|---|
| 1. 독립/종속·자료형 구분 | `output`이 정답(종속) |
| 2. 전처리 한 가지 이상 | `isnull`, `describe`, `StandardScaler` |
| 3. 분류 모델 학습·평가 | `train_test_split`, `fit`, `score`, `confusion_matrix` |

**준비물:** 같은 폴더의 `heart.csv` · scikit-learn  
**실행:** 위에서부터 ▶. 빈칸(`___`)과 `# TODO`만 채우면 됩니다.


## 0. 데이터 불러오기


In [ ]:
import pandas as pd

df = pd.read_csv('heart.csv')
print('행·열:', df.shape)
print('열 이름:', list(df.columns))
df.head()


> **힌트** 열 의미: `age`(나이), `sex`(성별), `cp`(흉통 유형), `trtbps`(안정시 혈압), `chol`(콜레스테롤),  
> `thalachh`(최대 심박수), … , **`output`(발병 여부 0/1 ← 우리가 맞히고 싶은 것)**


---

# 1. 데이터 속성 구분하기

독립 변수 = 모델에 **입력**하는 특징 / 종속 변수 = **맞히고 싶은 정답**


In [ ]:
# 자료형·고유값 개수를 보면 '수치형인지, 범주처럼 쓰는지' 감이 옵니다.
print(df.dtypes)
print('---')
print('output 값 분포:', df['output'].value_counts().to_dict())
print('sex 고유값:', df['sex'].unique())
print('cp 고유값:', df['cp'].unique())


> **생각하기 1** (활동지에 적어도 OK)
>
> | 구분 | 열 이름 예시 |
> |---|---|
> | 종속 변수 | `output` (하나!) |
> | 독립 변수 | `output`을 **뺀** 나머지 |
>
> `sex`, `cp`처럼 숫자는 숫자인데 **종류(범주)** 의미인 열이 있습니다. 어떤 열이 그런가요?
> →


---

# 2. 데이터 전처리하기

아래 중 **한 가지 이상**을 선택해 실행해 보세요.

| 전처리 | 왜? | 쓸 코드 힌트 |
|---|---|---|
| 결측치 확인 | 비어 있으면 학습이 안 되거나 이상해짐 | `df.isnull().sum()` |
| 이상치·분포 | 너무 큰/작은 값이 있는지 | `df.describe()` |
| 표준화 | 혈압·콜레스테롤 단위가 달라 k-NN에 유리 | `StandardScaler` |
| 핵심 속성 | 쓸 열만 고르기 | `df[['age','chol', ...]]` |


In [ ]:
# (1) 결측치 확인 — 전부 0이면 '결측 없음'으로 적어도 됩니다.
print(df.isnull().sum())


In [ ]:
# (2) 요약 통계로 이상치 감 잡기
df.describe()


In [ ]:
# (3) 독립/종속 나누기 + (선택) 표준화
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['output'])   # 종속 변수만 빼기
y = df['output']

# TODO: 표준화를 하려면 아래 주석을 푸세요.
# scaler = StandardScaler()
# X = scaler.fit_transform(X)   # 주의: 배열이 되어 열 이름이 사라집니다.

print('X 형태:', getattr(X, 'shape', None), 'y 개수:', len(y))


> **생각하기 2** 표준화를 **반드시** 해야 하는 모델과, 안 해도 비교적 괜찮은 모델이 있습니다.  
> k-NN / 의사결정트리 중 어떤 쪽이 거리(스케일)에 더 민감할까요?
> →


---

# 3. 분류 모델 선정과 학습

추천 순서: **훈련/시험 분할 → 모델 하나 고르기 → `fit` → `score` / 혼동행렬**

| 모델 | import 힌트 |
|---|---|
| k-NN | `from sklearn.neighbors import KNeighborsClassifier` |
| 로지스틱 회귀 | `from sklearn.linear_model import LogisticRegression` |
| 의사결정트리 | `from sklearn.tree import DecisionTreeClassifier` |
| SVM | `from sklearn.svm import SVC` |


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report

# 8:2 분할 (비율을 바꿔 비교해도 좋습니다)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7, stratify=y
)
print('훈련:', X_train.shape, '시험:', X_test.shape)


In [ ]:
# TODO: 모델을 하나 고르세요. (예시는 k-NN / 아래 줄 중 하나만 사용)
model = KNeighborsClassifier(n_neighbors=5)
# model = DecisionTreeClassifier(random_state=7, max_depth=4)

model.fit(X_train, y_train)

print('훈련 정확도:', round(model.score(X_train, y_train), 3))
print('시험 정확도:', round(model.score(X_test, y_test), 3))

y_pred = model.predict(X_test)
print('혼동행렬:\n', confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


> **생각하기 3** 시험 정확도만 보고 "좋은 모델"이라고 말할 수 있을까요?  
> 혼동행렬에서 **실제로 발병(1)인데 아니라고 예측**한 칸은 어떤 의미인가요?
> →

---

### 제출·정리 체크
- [ ] 종속 변수가 `output`임을 활동지에 적음
- [ ] 전처리 한 가지 이상 실행·결과 기록
- [ ] 모델 이름 + 시험 정확도 + 혼동행렬 해석 한 줄
